In [1]:
import pandas as pd
import os
from glob import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import CategoricalAccuracy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json
import random

In [2]:
directory_codeStructure = "D:\\ICSE_Dataset\\Derived_Features\\code_structure_feature_buggy_correct"
directory_embedding = "D:\\ICSE_Dataset\\Derived_Features\\embedding_features_buggy_correct"
directory_graph = "D:\\ICSE_Dataset\\Derived_Features\\graph_features_buggy_correct"

Label_1: Hyperparameter

In [16]:
#Embedding
def json_files_to_df(directory, label_keyword):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r') as f:
                    json_data = json.load(f)
                    data.append(json_data)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    df = pd.DataFrame(data)
    return df


df_hyperparameter_embedding = json_files_to_df(directory_embedding, 'Hyperparameter')
df_correct_embedding = json_files_to_df(directory_embedding, 'correct')
df_hyperparameter_embedding['label'] = 'Hyperparameter'
df_correct_embedding['label'] = 'correct'
min_count_embedding = min(len(df_hyperparameter_embedding), len(df_correct_embedding))
df_hyperparameter_embedding = df_hyperparameter_embedding.sample(n=min_count_embedding, random_state=1)
df_correct_embedding = df_correct_embedding.sample(n=min_count_embedding, random_state=1)
df_combined_embedding = pd.concat([df_hyperparameter_embedding, df_correct_embedding])
df_combined_embedding = df_combined_embedding.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_hyperparameter_embedding)} Hyperparameter files from embedding directory')
print(f'Selected {len(df_correct_embedding)} correct files from embedding directory')
print(f'Total combined data points from embedding directory: {len(df_combined_embedding)}')
df_combined_embedding.to_json('combined_embedding.json', orient='records')
print(df_combined_embedding)

Error decoding JSON from file D:\ICSE_Dataset\Derived_Features\embedding_features_buggy_correct\0_31556268_Hyperparameter_buggy.h5.json: Expecting value: line 1 column 1 (char 0)
Selected 4081 Hyperparameter files from embedding directory
Selected 4081 correct files from embedding directory
Total combined data points from embedding directory: 8162
                                 model_file  \
0     7657_68692047_Hyperparameter_buggy.h5   
1                  3639_50481178_correct.h5   
2     4523_71351646_Hyperparameter_buggy.h5   
3                  3969_61553510_correct.h5   
4      670_48934338_Hyperparameter_buggy.h5   
...                                     ...   
8157               1919_46642627_correct.h5   
8158               1415_77416892_correct.h5   
8159  4554_71351646_Hyperparameter_buggy.h5   
8160  5707_73765263_Hyperparameter_buggy.h5   
8161                453_68103873_correct.h5   

                                              embedding           label  
0     [0.16

In [18]:
labbeled_hyperparameter_embedding= df_combined_embedding.copy()
labbeled_hyperparameter_embedding.to_csv('labeled_hyperparameter_embedding.csv', index=False)

In [19]:
#graph
df_hyperparameter_graph = json_files_to_df(directory_graph, 'Hyperparameter')
df_correct_graph = json_files_to_df(directory_graph, 'correct')
df_hyperparameter_graph['label'] = 'Hyperparameter'
df_correct_graph['label'] = 'correct'
min_count_graph = min(len(df_hyperparameter_graph), len(df_correct_graph))
df_hyperparameter_graph = df_hyperparameter_graph.sample(n=min_count_graph, random_state=1)

df_correct_graph = df_correct_graph.sample(n=min_count_graph, random_state=1)
df_combined_graph = pd.concat([df_hyperparameter_graph, df_correct_graph])
df_combined_graph = df_combined_graph.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_hyperparameter_graph)} Hyperparameter files from graph directory')
print(f'Selected {len(df_correct_graph)} correct files from graph directory')
print(f'Total combined data points from graph directory: {len(df_combined_graph)}')

df_combined_graph.to_json('combined_graph.json', orient='records')
print(df_combined_graph)

labbeled_hyperparameter_graph= df_combined_graph.copy()
labbeled_hyperparameter_graph.to_csv('labeled_hyperparameter_graph.csv', index=False)

Selected 4081 Hyperparameter files from graph directory
Selected 4081 correct files from graph directory
Total combined data points from graph directory: 8162
                                      degree_centrality  \
0                                  {'0': 1.0, '1': 1.0}   
1     {'0': 0.2, '1': 0.4, '2': 0.4, '3': 0.4, '4': ...   
2     {'0': 0.2, '1': 0.4, '2': 0.4, '3': 0.4, '4': ...   
3                                  {'0': 1.0, '1': 1.0}   
4     {'0': 0.3333333333333333, '1': 0.6666666666666...   
...                                                 ...   
8157                     {'0': 0.5, '1': 1.0, '2': 0.5}   
8158                     {'0': 0.5, '1': 1.0, '2': 0.5}   
8159                               {'0': 1.0, '1': 1.0}   
8160                               {'0': 1.0, '1': 1.0}   
8161  {'0': 0.3333333333333333, '1': 0.6666666666666...   

                                   closeness_centrality  \
0                                  {'0': 0.0, '1': 1.0}   
1     {'0': 0.

In [3]:
import os
import json
import pandas as pd

def read_and_transpose_json(file_path):
    with open(file_path, 'r') as f:
        json_data = [json.loads(line) for line in f]
    df = pd.DataFrame(json_data).T 
    return df

def process_directory(directory, label_keyword):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_and_transpose_json(file_path)
                df_flat = df.values.flatten()  
                data.append(df_flat)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return pd.DataFrame(data)


directory_codeStructure = "D:\\ICSE_Dataset\\Derived_Features\\code_structure_feature_buggy_correct"
directory_embedding = "D:\\ICSE_Dataset\\Derived_Features\\embedding_feature_buggy_correct"
directory_graph = "D:\\ICSE_Dataset\\Derived_Features\\graph_feature_buggy_correct"
df_hyperparameter_codeStructure = process_directory(directory_codeStructure, 'Hyperparameter')
df_correct_codeStructure = process_directory(directory_codeStructure, 'correct')
df_hyperparameter_codeStructure['label'] = 'Hyperparameter'
df_correct_codeStructure['label'] = 'correct'
df_combined_codeStructure = pd.concat([df_hyperparameter_codeStructure, df_correct_codeStructure])
df_combined_codeStructure = df_combined_codeStructure.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Total combined data points from code structure directory: {len(df_combined_codeStructure)}')
df_combined_codeStructure.to_json('labeled_hyperparameter_codeStructure.json', orient='records')
print(df_combined_codeStructure)
labeled_hyperparameter_codeStructure = df_combined_codeStructure.copy()
labeled_hyperparameter_codeStructure.to_csv('labeled_hyperparameter_codeStructure.csv', index=False)


Total combined data points from code structure directory: 8289
             0          1              2             3           4    \
0     InputLayer       LSTM           LSTM  RepeatVector        LSTM   
1     InputLayer       LSTM          Dense         Dense           0   
2     InputLayer     Conv2D     Activation        Conv2D  Activation   
3     InputLayer       LSTM          Dense             0           1   
4     InputLayer      Dense          Dense         Dense           0   
...          ...        ...            ...           ...         ...   
8284  InputLayer       LSTM           LSTM         Dense           0   
8285  InputLayer      Dense          Dense         Dense           0   
8286  InputLayer  Embedding  Bidirectional         Dense     Dropout   
8287  InputLayer      Dense          Dense             0           1   
8288  InputLayer      Dense          Dense       Dropout       Dense   

               5                6           7           8           9   

In [4]:
labeled_hyperparameter_codeStructure

,0,1,2,3,4,5,6,7,8,9,...,278,279,280,281,282,283,284,285,286,287
0,InputLayer,LSTM,LSTM,RepeatVector,LSTM,LSTM,TimeDistributed,0,1,2,...,None,None,None,None,None,None,None,None,None,None
1,InputLayer,LSTM,Dense,Dense,0,1,2,3,None,InputLayer,...,None,None,None,None,None,None,None,None,None,None
2,InputLayer,Conv2D,Activation,Conv2D,Activation,MaxPooling2D,Flatten,Dense,Activation,Dense,...,None,None,None,None,None,None,None,None,None,None
3,InputLayer,LSTM,Dense,0,1,2,None,InputLayer,LSTM,LSTM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,InputLayer,Dense,Dense,Dense,0,1,2,3,None,InputLayer,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284,InputLayer,LSTM,LSTM,Dense,0,1,2,3,None,InputLayer,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8285,InputLayer,Dense,Dense,Dense,0,1,2,3,None,InputLayer,...,None,None,None,None,None,None,None,None,None,None
8286,InputLayer,Embedding,Bidirectional,Dense,Dropout,Dense,0,1,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8287,InputLayer,Dense,Dense,0,1,2,None,InputLayer,Dense,Dense,...,None,None,None,None,None,None,None,None,None,None


In [5]:
def read_json(file_path):
    with open(file_path, 'r') as f:
        json_data = [json.loads(line) for line in f]
    df = pd.DataFrame(json_data)
    return df

def determine_max_rows(directory, label_keyword):
    max_rows = 0
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                max_rows = max(max_rows, len(df))
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return max_rows
def process_directory(directory, label_keyword, max_rows):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                current_rows = len(df)
                if current_rows < max_rows:
                    padding = pd.DataFrame([{}] * (max_rows - current_rows))
                    df = pd.concat([df, padding], ignore_index=True)
                df_flat = df.values.flatten() 
                data.append(df_flat)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return pd.DataFrame(data)

max_rows_codeStructure = determine_max_rows(directory_codeStructure, 'Hyperparameter')
max_rows_codeStructure = max(max_rows_codeStructure, determine_max_rows(directory_codeStructure, 'correct'))
print(f'Maximum number of rows for padding: {max_rows_codeStructure}')
df_hyperparameter_codeStructure = process_directory(directory_codeStructure, 'Hyperparameter', max_rows_codeStructure)
df_correct_codeStructure = process_directory(directory_codeStructure, 'correct', max_rows_codeStructure)
df_hyperparameter_codeStructure['label'] = 'Hyperparameter'
df_correct_codeStructure['label'] = 'correct'

df_combined_codeStructure = pd.concat([df_hyperparameter_codeStructure, df_correct_codeStructure])
df_combined_codeStructure = df_combined_codeStructure.sample(frac=1, random_state=1).reset_index(drop=True)

print(f'Total combined data points from code structure directory: {len(df_combined_codeStructure)}')
df_combined_codeStructure.to_json('labeled_hyperparameter_codeStructure.json', orient='records')
print(df_combined_codeStructure)
labeled_hyperparameter_codeStructure_padded = df_combined_codeStructure.copy()
labeled_hyperparameter_codeStructure_padded.to_csv('labeled_hyperparameter_codeStructure_padding.csv', index=False)


Maximum number of rows for padding: 24
Total combined data points from code structure directory: 8289
               0    1     2          3     4     5     6                    7  \
0     InputLayer  0.0  None       LSTM  None  None  None        [None, 10, 5]   
1     InputLayer  0.0  None       LSTM  None  None  None         [None, 2, 2]   
2     InputLayer  0.0  None     Conv2D  None  None  None  [None, 100, 100, 1]   
3     InputLayer  0.0  None       LSTM  None  None  None     [None, 10000, 1]   
4     InputLayer  0.0  None      Dense  None  None  None          [None, 784]   
...          ...  ...   ...        ...   ...   ...   ...                  ...   
8284  InputLayer  0.0  None       LSTM  None  None  None         [None, 1, 1]   
8285  InputLayer  0.0  None      Dense  None  None  None            [None, 2]   
8286  InputLayer  0.0  None  Embedding  None  None  None           [None, 30]   
8287  InputLayer  0.0  None      Dense  None  None  None            [None, 2]   
8288  I

In [6]:
labeled_hyperparameter_codeStructure_padded

,0,1,2,3,4,5,6,7,8,9,...,279,280,281,282,283,284,285,286,287,label
0,InputLayer,0.0,None,LSTM,None,None,None,"[None, 10, 5]","[None, 10, 8]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
1,InputLayer,0.0,None,LSTM,None,None,None,"[None, 2, 2]","[None, 100]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
2,InputLayer,0.0,None,Conv2D,None,None,None,"[None, 100, 100, 1]","[None, 98, 98, 32]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
3,InputLayer,0.0,None,LSTM,None,None,None,"[None, 10000, 1]","[None, 50]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hyperparameter
4,InputLayer,0.0,None,Dense,None,None,None,"[None, 784]","[None, 784]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284,InputLayer,0.0,None,LSTM,None,None,None,"[None, 1, 1]","[None, 1, 64]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hyperparameter
8285,InputLayer,0.0,None,Dense,None,None,None,"[None, 2]","[None, 2]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
8286,InputLayer,0.0,None,Embedding,None,None,None,"[None, 30]","[None, 30, 300]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hyperparameter
8287,InputLayer,0.0,None,Dense,None,None,None,"[None, 2]","[None, 2]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct


In [7]:
# print one row of the dataframe
print(labeled_hyperparameter_codeStructure_padded.iloc[0])

0        InputLayer
1               0.0
2              None
3              LSTM
4              None
            ...    
284             NaN
285             NaN
286             NaN
287             NaN
label       correct
Name: 0, Length: 289, dtype: object


Label 2: Layer

In [3]:
#Embedding
def json_files_to_df(directory, label_keyword):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r') as f:
                    json_data = json.load(f)
                    data.append(json_data)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    df = pd.DataFrame(data)
    return df


df_Layer_embedding = json_files_to_df(directory_embedding, 'Layer')
df_correct_embedding = json_files_to_df(directory_embedding, 'correct')
df_Layer_embedding['label'] = 'Layer'
df_correct_embedding['label'] = 'correct'
min_count_embedding = min(len(df_Layer_embedding), len(df_correct_embedding))
df_Layer_embedding = df_Layer_embedding.sample(n=min_count_embedding, random_state=1)
df_correct_embedding = df_correct_embedding.sample(n=min_count_embedding, random_state=1)
df_combined_embedding = pd.concat([df_Layer_embedding, df_correct_embedding])
df_combined_embedding = df_combined_embedding.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_Layer_embedding)} Layer files from embedding directory')
print(f'Selected {len(df_correct_embedding)} correct files from embedding directory')
print(f'Total combined data points from embedding directory: {len(df_combined_embedding)}')
df_combined_embedding.to_json('combined_embedding.json', orient='records')
print(df_combined_embedding)
labbeled_Layer_embedding= df_combined_embedding.copy()
labbeled_Layer_embedding.to_csv('labeled_Layer_embedding.csv', index=False)

#Graph
df_Layer_graph = json_files_to_df(directory_graph, 'Layer')
df_correct_graph = json_files_to_df(directory_graph, 'correct')
df_Layer_graph['label'] = 'Layer'
df_correct_graph['label'] = 'correct'
min_count_graph = min(len(df_Layer_graph), len(df_correct_graph))
df_Layer_graph = df_Layer_graph.sample(n=min_count_graph, random_state=1)

df_correct_graph = df_correct_graph.sample(n=min_count_graph, random_state=1)
df_combined_graph = pd.concat([df_Layer_graph, df_correct_graph])
df_combined_graph = df_combined_graph.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_Layer_graph)} Layer files from graph directory')
print(f'Selected {len(df_correct_graph)} correct files from graph directory')
print(f'Total combined data points from graph directory: {len(df_combined_graph)}')

df_combined_graph.to_json('combined_graph.json', orient='records')
print(df_combined_graph)

labbeled_Layer_graph= df_combined_graph.copy()
labbeled_Layer_graph.to_csv('labeled_Layer_graph.csv', index=False)

#Code Structure
def read_json(file_path):
    with open(file_path, 'r') as f:
        json_data = [json.loads(line) for line in f]
    df = pd.DataFrame(json_data)
    return df

def determine_max_rows(directory, label_keyword):
    max_rows = 0
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                max_rows = max(max_rows, len(df))
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return max_rows
def process_directory(directory, label_keyword, max_rows):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                current_rows = len(df)
                if current_rows < max_rows:
                    padding = pd.DataFrame([{}] * (max_rows - current_rows))
                    df = pd.concat([df, padding], ignore_index=True)
                df_flat = df.values.flatten() 
                data.append(df_flat)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return pd.DataFrame(data)

max_rows_codeStructure = determine_max_rows(directory_codeStructure, 'Layer')
max_rows_codeStructure = max(max_rows_codeStructure, determine_max_rows(directory_codeStructure, 'correct'))
print(f'Maximum number of rows for padding: {max_rows_codeStructure}')
df_Layer_codeStructure = process_directory(directory_codeStructure, 'Layer', max_rows_codeStructure)
df_correct_codeStructure = process_directory(directory_codeStructure, 'correct', max_rows_codeStructure)
df_Layer_codeStructure['label'] = 'Layer'
df_correct_codeStructure['label'] = 'correct'

df_combined_codeStructure = pd.concat([df_Layer_codeStructure, df_correct_codeStructure])
df_combined_codeStructure = df_combined_codeStructure.sample(frac=1, random_state=1).reset_index(drop=True)

print(f'Total combined data points from code structure directory: {len(df_combined_codeStructure)}')
df_combined_codeStructure.to_json('labeled_Layer_codeStructure.json', orient='records')
print(df_combined_codeStructure)
labeled_Layer_codeStructure_padded = df_combined_codeStructure.copy()
labeled_Layer_codeStructure_padded.to_csv('labeled_Layer_codeStructure_padding.csv', index=False)

Selected 1371 Layer files from embedding directory
Selected 1371 correct files from embedding directory
Total combined data points from embedding directory: 2742
                        model_file  \
0         3695_73765263_correct.h5   
1     5167_72328867_Layer_buggy.h5   
2     3259_46642627_Layer_buggy.h5   
3          892_70848143_correct.h5   
4      545_50481178_Layer_buggy.h5   
...                            ...   
2737  3916_66840108_Layer_buggy.h5   
2738  2566_77416892_Layer_buggy.h5   
2739   265_44164749_Layer_buggy.h5   
2740  5319_73146829_Layer_buggy.h5   
2741  1982_70848143_Layer_buggy.h5   

                                              embedding    label  
0     [0.1762203425168991, 0.2688315808773041, 0.105...  correct  
1     [0.12099839746952057, 0.23035728931427002, 0.0...    Layer  
2     [0.18868303298950195, 0.25621867179870605, 0.0...    Layer  
3     [0.1630043089389801, 0.23289208114147186, 0.08...  correct  
4     [0.18807436525821686, 0.2614184021949768

Label 3: Optimization

In [4]:
def json_files_to_df(directory, label_keyword):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r') as f:
                    json_data = json.load(f)
                    data.append(json_data)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    df = pd.DataFrame(data)
    return df

# Embedding
df_Optimization_embedding = json_files_to_df(directory_embedding, 'Optimization')
df_correct_embedding = json_files_to_df(directory_embedding, 'correct')
df_Optimization_embedding['label'] = 'Optimization'
df_correct_embedding['label'] = 'correct'
min_count_embedding = min(len(df_Optimization_embedding), len(df_correct_embedding))
df_Optimization_embedding = df_Optimization_embedding.sample(n=min_count_embedding, random_state=1)
df_correct_embedding = df_correct_embedding.sample(n=min_count_embedding, random_state=1)
df_combined_embedding = pd.concat([df_Optimization_embedding, df_correct_embedding])
df_combined_embedding = df_combined_embedding.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_Optimization_embedding)} Optimization files from embedding directory')
print(f'Selected {len(df_correct_embedding)} correct files from embedding directory')
print(f'Total combined data points from embedding directory: {len(df_combined_embedding)}')
df_combined_embedding.to_json('combined_embedding.json', orient='records')
print(df_combined_embedding)
labeled_Optimization_embedding = df_combined_embedding.copy()
labeled_Optimization_embedding.to_csv('labeled_Optimization_embedding.csv', index=False)

# Graph
df_Optimization_graph = json_files_to_df(directory_graph, 'Optimization')
df_correct_graph = json_files_to_df(directory_graph, 'correct')
df_Optimization_graph['label'] = 'Optimization'
df_correct_graph['label'] = 'correct'
min_count_graph = min(len(df_Optimization_graph), len(df_correct_graph))
df_Optimization_graph = df_Optimization_graph.sample(n=min_count_graph, random_state=1)
df_correct_graph = df_correct_graph.sample(n=min_count_graph, random_state=1)
df_combined_graph = pd.concat([df_Optimization_graph, df_correct_graph])
df_combined_graph = df_combined_graph.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_Optimization_graph)} Optimization files from graph directory')
print(f'Selected {len(df_correct_graph)} correct files from graph directory')
print(f'Total combined data points from graph directory: {len(df_combined_graph)}')
df_combined_graph.to_json('combined_graph.json', orient='records')
print(df_combined_graph)
labeled_Optimization_graph = df_combined_graph.copy()
labeled_Optimization_graph.to_csv('labeled_Optimization_graph.csv', index=False)

# Code Structure
def read_json(file_path):
    with open(file_path, 'r') as f:
        json_data = [json.loads(line) for line in f]
    df = pd.DataFrame(json_data)
    return df

def determine_max_rows(directory, label_keyword):
    max_rows = 0;
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                max_rows = max(max_rows, len(df))
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return max_rows

def process_directory(directory, label_keyword, max_rows):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                current_rows = len(df)
                if current_rows < max_rows:
                    padding = pd.DataFrame([{}] * (max_rows - current_rows))
                    df = pd.concat([df, padding], ignore_index=True)
                df_flat = df.values.flatten() 
                data.append(df_flat)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return pd.DataFrame(data)

max_rows_codeStructure = determine_max_rows(directory_codeStructure, 'Optimization')
max_rows_codeStructure = max(max_rows_codeStructure, determine_max_rows(directory_codeStructure, 'correct'))
print(f'Maximum number of rows for padding: {max_rows_codeStructure}')
df_Optimization_codeStructure = process_directory(directory_codeStructure, 'Optimization', max_rows_codeStructure)
df_correct_codeStructure = process_directory(directory_codeStructure, 'correct', max_rows_codeStructure)
df_Optimization_codeStructure['label'] = 'Optimization'
df_correct_codeStructure['label'] = 'correct'

df_combined_codeStructure = pd.concat([df_Optimization_codeStructure, df_correct_codeStructure])
df_combined_codeStructure = df_combined_codeStructure.sample(frac=1, random_state=1).reset_index(drop=True)

print(f'Total combined data points from code structure directory: {len(df_combined_codeStructure)}')
df_combined_codeStructure.to_json('labeled_Optimization_codeStructure.json', orient='records')
print(df_combined_codeStructure)
labeled_Optimization_codeStructure_padded = df_combined_codeStructure.copy()
labeled_Optimization_codeStructure_padded.to_csv('labeled_Optimization_codeStructure_padding.csv', index=False)

Selected 1239 Optimization files from embedding directory
Selected 1239 correct files from embedding directory
Total combined data points from embedding directory: 2478
                               model_file  \
0                1997_50481178_correct.h5   
1                2055_52782432_correct.h5   
2                2130_59325381_correct.h5   
3     7847_37624102_Optimization_buggy.h5   
4                2396_48934338_correct.h5   
...                                   ...   
2473  3249_31556268_Optimization_buggy.h5   
2474  2585_67727890_Optimization_buggy.h5   
2475  6251_77416892_Optimization_buggy.h5   
2476  6882_70178206_Optimization_buggy.h5   
2477  7852_37624102_Optimization_buggy.h5   

                                              embedding         label  
0     [0.19117514789104462, 0.25849002599716187, 0.1...       correct  
1     [0.1671713888645172, 0.2590924799442291, 0.114...       correct  
2     [0.14262327551841736, 0.2287479043006897, 0.09...       correct  
3 

Label 4: Loss

In [5]:
def json_files_to_df(directory, label_keyword):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r') as f:
                    json_data = json.load(f)
                    data.append(json_data)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    df = pd.DataFrame(data)
    return df

# Embedding
df_Loss_embedding = json_files_to_df(directory_embedding, 'Loss')
df_correct_embedding = json_files_to_df(directory_embedding, 'correct')
df_Loss_embedding['label'] = 'Loss'
df_correct_embedding['label'] = 'correct'
min_count_embedding = min(len(df_Loss_embedding), len(df_correct_embedding))
df_Loss_embedding = df_Loss_embedding.sample(n=min_count_embedding, random_state=1)
df_correct_embedding = df_correct_embedding.sample(n=min_count_embedding, random_state=1)
df_combined_embedding = pd.concat([df_Loss_embedding, df_correct_embedding])
df_combined_embedding = df_combined_embedding.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_Loss_embedding)} Loss files from embedding directory')
print(f'Selected {len(df_correct_embedding)} correct files from embedding directory')
print(f'Total combined data points from embedding directory: {len(df_combined_embedding)}')
df_combined_embedding.to_json('combined_embedding.json', orient='records')
print(df_combined_embedding)
labeled_Loss_embedding = df_combined_embedding.copy()
labeled_Loss_embedding.to_csv('labeled_Loss_embedding.csv', index=False)

# Graph
df_Loss_graph = json_files_to_df(directory_graph, 'Loss')
df_correct_graph = json_files_to_df(directory_graph, 'correct')
df_Loss_graph['label'] = 'Loss'
df_correct_graph['label'] = 'correct'
min_count_graph = min(len(df_Loss_graph), len(df_correct_graph))
df_Loss_graph = df_Loss_graph.sample(n=min_count_graph, random_state=1)
df_correct_graph = df_correct_graph.sample(n=min_count_graph, random_state=1)
df_combined_graph = pd.concat([df_Loss_graph, df_correct_graph])
df_combined_graph = df_combined_graph.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_Loss_graph)} Loss files from graph directory')
print(f'Selected {len(df_correct_graph)} correct files from graph directory')
print(f'Total combined data points from graph directory: {len(df_combined_graph)}')
df_combined_graph.to_json('combined_graph.json', orient='records')
print(df_combined_graph)
labeled_Loss_graph = df_combined_graph.copy()
labeled_Loss_graph.to_csv('labeled_Loss_graph.csv', index=False)

# Code Structure
def read_json(file_path):
    with open(file_path, 'r') as f:
        json_data = [json.loads(line) for line in f]
    df = pd.DataFrame(json_data)
    return df

def determine_max_rows(directory, label_keyword):
    max_rows = 0;
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                max_rows = max(max_rows, len(df))
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return max_rows

def process_directory(directory, label_keyword, max_rows):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                current_rows = len(df)
                if current_rows < max_rows:
                    padding = pd.DataFrame([{}] * (max_rows - current_rows))
                    df = pd.concat([df, padding], ignore_index=True)
                df_flat = df.values.flatten() 
                data.append(df_flat)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return pd.DataFrame(data)

max_rows_codeStructure = determine_max_rows(directory_codeStructure, 'Loss')
max_rows_codeStructure = max(max_rows_codeStructure, determine_max_rows(directory_codeStructure, 'correct'))
print(f'Maximum number of rows for padding: {max_rows_codeStructure}')
df_Loss_codeStructure = process_directory(directory_codeStructure, 'Loss', max_rows_codeStructure)
df_correct_codeStructure = process_directory(directory_codeStructure, 'correct', max_rows_codeStructure)
df_Loss_codeStructure['label'] = 'Loss'
df_correct_codeStructure['label'] = 'correct'

df_combined_codeStructure = pd.concat([df_Loss_codeStructure, df_correct_codeStructure])
df_combined_codeStructure = df_combined_codeStructure.sample(frac=1, random_state=1).reset_index(drop=True)

print(f'Total combined data points from code structure directory: {len(df_combined_codeStructure)}')
df_combined_codeStructure.to_json('labeled_Loss_codeStructure.json', orient='records')
print(df_combined_codeStructure)
labeled_Loss_codeStructure_padded = df_combined_codeStructure.copy()
labeled_Loss_codeStructure_padded.to_csv('labeled_Loss_codeStructure_padding.csv', index=False)


Selected 2147 Loss files from embedding directory
Selected 2147 correct files from embedding directory
Total combined data points from embedding directory: 4294
                       model_file  \
0        3888_61553510_correct.h5   
1        1956_50481178_correct.h5   
2     3050_68716219_Loss_buggy.h5   
3     1712_31556268_Loss_buggy.h5   
4        3710_52782432_correct.h5   
...                           ...   
4289      139_66952606_correct.h5   
4290      768_70406438_correct.h5   
4291   958_48934338_Loss_buggy.h5   
4292     2254_64522751_correct.h5   
4293  4094_70777445_Loss_buggy.h5   

                                              embedding    label  
0     [0.16860774159431458, 0.26869839429855347, 0.1...  correct  
1     [0.1863144040107727, 0.26561617851257324, 0.10...  correct  
2     [0.1786985695362091, 0.22215530276298523, 0.09...     Loss  
3     [0.17462360858917236, 0.24255846440792084, 0.0...     Loss  
4     [0.17078107595443726, 0.26121094822883606, 0.1...  co

Label 5: Activation


In [6]:
def json_files_to_df(directory, label_keyword):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r') as f:
                    json_data = json.load(f)
                    data.append(json_data)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    df = pd.DataFrame(data)
    return df

# Embedding
df_Activation_embedding = json_files_to_df(directory_embedding, 'Activation')
df_correct_embedding = json_files_to_df(directory_embedding, 'correct')
df_Activation_embedding['label'] = 'Activation'
df_correct_embedding['label'] = 'correct'
min_count_embedding = min(len(df_Activation_embedding), len(df_correct_embedding))
df_Activation_embedding = df_Activation_embedding.sample(n=min_count_embedding, random_state=1)
df_correct_embedding = df_correct_embedding.sample(n=min_count_embedding, random_state=1)
df_combined_embedding = pd.concat([df_Activation_embedding, df_correct_embedding])
df_combined_embedding = df_combined_embedding.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_Activation_embedding)} Activation files from embedding directory')
print(f'Selected {len(df_correct_embedding)} correct files from embedding directory')
print(f'Total combined data points from embedding directory: {len(df_combined_embedding)}')
df_combined_embedding.to_json('combined_embedding.json', orient='records')
print(df_combined_embedding)
labeled_Activation_embedding = df_combined_embedding.copy()
labeled_Activation_embedding.to_csv('labeled_Activation_embedding.csv', index=False)

# Graph
df_Activation_graph = json_files_to_df(directory_graph, 'Activation')
df_correct_graph = json_files_to_df(directory_graph, 'correct')
df_Activation_graph['label'] = 'Activation'
df_correct_graph['label'] = 'correct'
min_count_graph = min(len(df_Activation_graph), len(df_correct_graph))
df_Activation_graph = df_Activation_graph.sample(n=min_count_graph, random_state=1)
df_correct_graph = df_correct_graph.sample(n=min_count_graph, random_state=1)
df_combined_graph = pd.concat([df_Activation_graph, df_correct_graph])
df_combined_graph = df_combined_graph.sample(frac=1, random_state=1).reset_index(drop=True)
print(f'Selected {len(df_Activation_graph)} Activation files from graph directory')
print(f'Selected {len(df_correct_graph)} correct files from graph directory')
print(f'Total combined data points from graph directory: {len(df_combined_graph)}')
df_combined_graph.to_json('combined_graph.json', orient='records')
print(df_combined_graph)
labeled_Activation_graph = df_combined_graph.copy()
labeled_Activation_graph.to_csv('labeled_Activation_graph.csv', index=False)

# Code Structure
def read_json(file_path):
    with open(file_path, 'r') as f:
        json_data = [json.loads(line) for line in f]
    df = pd.DataFrame(json_data)
    return df

def determine_max_rows(directory, label_keyword):
    max_rows = 0;
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                max_rows = max(max_rows, len(df))
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return max_rows

def process_directory(directory, label_keyword, max_rows):
    data = []
    for filename in os.listdir(directory):
        if label_keyword in filename:
            file_path = os.path.join(directory, filename)
            try:
                df = read_json(file_path)
                current_rows = len(df)
                if current_rows < max_rows:
                    padding = pd.DataFrame([{}] * (max_rows - current_rows))
                    df = pd.concat([df, padding], ignore_index=True)
                df_flat = df.values.flatten() 
                data.append(df_flat)
            except json.JSONDecodeError as e:
                print(f'Error decoding JSON from file {file_path}: {e}')
            except Exception as e:
                print(f'Unexpected error reading file {file_path}: {e}')
    return pd.DataFrame(data)

max_rows_codeStructure = determine_max_rows(directory_codeStructure, 'Activation')
max_rows_codeStructure = max(max_rows_codeStructure, determine_max_rows(directory_codeStructure, 'correct'))
print(f'Maximum number of rows for padding: {max_rows_codeStructure}')
df_Activation_codeStructure = process_directory(directory_codeStructure, 'Activation', max_rows_codeStructure)
df_correct_codeStructure = process_directory(directory_codeStructure, 'correct', max_rows_codeStructure)
df_Activation_codeStructure['label'] = 'Activation'
df_correct_codeStructure['label'] = 'correct'

df_combined_codeStructure = pd.concat([df_Activation_codeStructure, df_correct_codeStructure])
df_combined_codeStructure = df_combined_codeStructure.sample(frac=1, random_state=1).reset_index(drop=True)

print(f'Total combined data points from code structure directory: {len(df_combined_codeStructure)}')
df_combined_codeStructure.to_json('labeled_Activation_codeStructure.json', orient='records')
print(df_combined_codeStructure)
labeled_Activation_codeStructure_padded = df_combined_codeStructure.copy()
labeled_Activation_codeStructure_padded.to_csv('labeled_Activation_codeStructure_padding.csv', index=False)


Selected 311 Activation files from embedding directory
Selected 311 correct files from embedding directory
Total combined data points from embedding directory: 622
                            model_file  \
0             2068_46642627_correct.h5   
1             2348_66840108_correct.h5   
2    6362_50481178_Activation_buggy.h5   
3             1611_34311586_correct.h5   
4    4664_73148058_Activation_buggy.h5   
..                                 ...   
617  2008_67590787_Activation_buggy.h5   
618  4658_73148058_Activation_buggy.h5   
619  2096_67590787_Activation_buggy.h5   
620  4715_73148058_Activation_buggy.h5   
621  4785_73148058_Activation_buggy.h5   

                                             embedding       label  
0    [0.18680569529533386, 0.2601884603500366, 0.09...     correct  
1    [0.18907904624938965, 0.25772982835769653, 0.0...     correct  
2    [0.18763253092765808, 0.26183465123176575, 0.0...  Activation  
3    [0.1823645532131195, 0.24697138369083405, 0.09... 